# Fetch the Milk

*A hands-on introduction to the Entity Query Language (EQL), told as one robot's morning.*

A PR2 wakes up in a kitchen and wants to serve breakfast. To do that it must answer, by itself, questions you answer without noticing: *Is there milk? What is it inside of? How do I open that? Where should I stand?* And when the morning is over: *what do I remember about it?*

You will answer each of these questions by writing queries, one small step at a time.

**How this works**

- Run every cell in order with `Shift+Enter`.
- ✏️ **Your turn** marks an exercise. The collapsed cell right below it always holds the solution: click the `…` to reveal it, or just run it to catch up. Try it yourself first.
- Hover any name to see its documentation, and completions pop up while you type. When a cell mentions a file, it is a link you can open.
- The kitchen lives in RViz, on the right. Look at it often — most queries change what you see there.

## 1 · Wake up

One cell boots the whole session: it starts ROS, loads the kitchen from its URDF description, and connects it to RViz. On a fresh session the first run also generates the database interfaces and takes about a minute — a good moment to peek at [tutorial/setup.py](tutorial/setup.py), where everything this notebook imports is defined.

In [ ]:
from tutorial.setup import *

A kitchen parsed from URDF is pure geometry: shapes, frames, joints. Two things are missing before breakfast can start. *Meaning* — it takes the world's reasoner to look at that geometry and recognize, for instance, which of the boxes is a fridge. And the milk itself — the shelf and carton that `stock_the_fridge` adds are modelled in [tutorial/kitchen.py](tutorial/kitchen.py), if you ever want the details.

In [ ]:
WorldReasoner(world).reason()
stock_the_fridge(world)

## 2 · Is there milk?

The world is a graph of ordinary Python objects, and EQL lets you ask for them the way you would ask a colleague: a quantifier, then what you want. `the(...)` means *exactly one of these exists — bring it to me*.

In [ ]:
milk = the(Milk).first()
milk.name

A name is proof, but not a *place*. `highlight` paints whatever a query found straight onto the simulated world — look for the glow in RViz.

In [ ]:
highlight(milk)

✏️ **Your turn.** Breakfast also needs the fridge. Get it the same way.

In [ ]:
fridge = ...

In [ ]:
fridge = the(Fridge).first()

In [ ]:
fridge.name

Asking by type only goes so far — usually you *describe* what you want. A query then has three parts: a **variable** that ranges over candidates, a **`where`** with your conditions, and the quantifier. The robot's gripper, for instance, cares about everything with a handle:

In [ ]:
query = an(body := variable(Body, domain=world.bodies)).where(contains(body.name.name, "handle"))
query.tolist()

Did it find the right ones? Don't read the names — look. Highlights stack, and each call can pick its own color, so every query can leave its mark.

In [ ]:
highlight(query, color=Color(0.2, 0.6, 1.0))

A query is a data structure, not a string — so before you run one, you can ask it to read itself back in English. Keep this trick in mind: it pays off more the bigger your queries get.

In [ ]:
verbalize_expression(query)

## 3 · What is the milk inside of?

The world can tell you the milk's coordinates — but `x=1.3, y=-0.6` is not understanding. The question that matters is *what contains it*, and that answer is stored nowhere: it has to be **computed from the shapes themselves**. When a query needs a computation EQL does not offer, you bring your own Python: decorate a function with `@symbolic_function` and it becomes usable inside queries — called on query variables it is deferred and evaluated once per candidate.

✏️ **Your turn.** Implement `is_inside(body, container)`. The building block is `InsideOf(body, container)()`, which returns the *fraction* of `body`'s volume lying inside `container` — `1.0` means fully contained, `0.0` not at all. Real meshes rarely reach exactly `1.0`, so accept anything above a high threshold like `0.9`.

In [ ]:
@symbolic_function
def is_inside(body: KinematicStructureEntity, container: KinematicStructureEntity) -> bool:
    """
    Decide whether ``body`` counts as being inside ``container``.
    """
    ...

In [ ]:
@symbolic_function
def is_inside(body: KinematicStructureEntity, container: KinematicStructureEntity) -> bool:
    """
    Decide whether ``body`` counts as being inside ``container``.
    """
    return InsideOf(body, container)() > 0.9

In [ ]:
assert is_inside(milk.root, fridge.root)
print("✔ the milk is indeed inside the fridge")

Of course the robot should not need us to tell it to check the fridge. Storage places carry an `IsStorageSpace` annotation, so it can search them itself.

✏️ **Your turn.** Complete the query: *the storage space that the milk is inside of*.

In [ ]:
storage_query = the(space := variable(IsStorageSpace)).where(...)

In [ ]:
storage_query = the(space := variable(IsStorageSpace)).where(is_inside(milk.root, space.root))

In [ ]:
storage = storage_query.first()
print(storage.name)

The robot worked out, on its own, that the milk is in the fridge. To get at it, it will have to pull the fridge open — which means finding the door's handle.

✏️ **Your turn.** Storage spaces know their `doors`. Starting from `storage`, type a `.` and let the completion popup guide you to the handle of the first door.

In [ ]:
handle = ...

In [ ]:
handle = storage.doors[0].handle

In [ ]:
handle.name

The robot's plan now hangs on a chain of three answers: the storage, its door, the handle. Paint the chain, each link in its own color — starting with the whole storage:

In [ ]:
highlight(storage, color=Color(1.0, 0.8, 0.0))

✏️ **Your turn.** Highlighting a body again repaints it — so refine: repaint the first door, then its handle.

In [ ]:
highlight(..., color=Color(0.0, 1.0, 1.0))
highlight(..., color=Color(1.0, 0.0, 1.0))

In [ ]:
highlight(storage.doors[0], color=Color(0.0, 1.0, 1.0))
highlight(handle, color=Color(1.0, 0.0, 1.0))

## 4 · Who taught the robot what a fridge is?

Storage, door, handle — that chain only worked because the world already knew what its doors and handles *are*. Nobody labeled them by hand: the reasoner from the first minute inferred every one of them, and its rules are ordinary EQL queries with one new piece. `inference(Handle)(root=body)` states a **conclusion** — *for every match, there is a `Handle`*. Here is the reasoner's first rule, rebuilt from parts you already know, read back before running it:

In [ ]:
clear_highlights()
body = variable(Body, domain=world.bodies)
handle_rule = an(inference(Handle)(root=body)).where(contains(body.name.name, "handle"))
verbalize_expression(handle_rule)

An *if–then* sentence: the `where` became the *if*, the conclusion the *then*. Now run it — every match comes back wrapped in a fresh `Handle`:

In [ ]:
inferred_handles = handle_rule.tolist()
highlight(inferred_handles, color=Color(0.2, 0.6, 1.0))

A conclusion never has to be taken on faith: everything a rule infers remembers *why* it exists. `explain_inference` hands back the conditions that fired, the values that satisfied them, and the exact line of the rule:

In [ ]:
explanation = explain_inference(inferred_handles[0])
print(explanation.as_string())

The explanation even carries the whole rule with it — hand anyone this `Handle`, and it can still tell them, in English, where it came from:

In [ ]:
verbalize_expression(explanation.query_root)

✏️ **Your turn.** One rung up the ladder: a **door** is a body that *swings* — a `RevoluteConnection` attaches it to its furniture — and *carries a handle* — a `FixedConnection` attaches the handle's body to it. Rules stack: this one consumes the `Handle` conclusions the reasoner already added to the world. The conclusion is written; give the rule its two conditions, one per connection — every connection has a `parent` and a `child`.

In [ ]:
a_handle = variable(Handle, domain=world.semantic_annotations)
swing = variable(RevoluteConnection, domain=world.connections)
mount = variable(FixedConnection, domain=world.connections)

In [ ]:
door_rule = an(inference(Door)(root=mount.parent, handle=a_handle)).where(
    ...,  # the mount carries the handle
    ...,  # the mounted body is the swinging one
)

In [ ]:
door_rule = an(inference(Door)(root=mount.parent, handle=a_handle)).where(
    mount.child == a_handle.root,
    mount.parent == swing.child,
)

In [ ]:
inferred_doors = door_rule.tolist()
highlight(inferred_doors, color=Color(0.2, 1.0, 0.2))

Every door in the kitchen glows green, each holding onto its handle. Read your rule back, and ask one of its doors to justify itself — this is exactly how the fridge door you are about to pull open earned its name. The reasoner's full rule set lives in [world_semantic_annotations_mcrdr_defs.py](https://github.com/AbdelrhmanBassiouny/cognitive_robot_abstract_machine/blob/ijcai-tutorial/semantic_digital_twin/src/semantic_digital_twin/reasoning/world_rdr/world_semantic_annotations_mcrdr_defs.py).

In [ ]:
print(verbalize_expression(door_rule), end="\n\n")
print(explain_inference(inferred_doors[0]).as_string())

## 5 · Where should the robot stand?

Everything so far *found* things that exist. But a good place to stand is not in the world — there is nothing to find. So EQL flips from finding to **generating**: write `a(...)` with `...` for every field you leave open, and a probabilistic model proposes values for them.

In [ ]:
clear_highlights()

guess = a(Point3)(x=..., y=..., z=..., reference_frame=handle.root)
guess.expression.limit(1000)
show_points(guess.tolist(backend=ProbabilisticBackend()))

Look at RViz: a cloud of green points floating through the whole kitchen — walls, furniture, thin air. The model has no idea what the points are *for*. Constraining it is your job, one fact at a time.

✏️ **Your turn.** Robots stand on the floor. The points are expressed in the handle's frame, so the floor lies at *minus the handle's height*. Pin `z` there and leave `x` and `y` free.

In [ ]:
floor_z = ...  # hint: the handle's height above the floor is handle.root.global_pose.z

In [ ]:
floor_z = -float(handle.root.global_pose.z)

In [ ]:
on_floor = a(Point3)(x=..., y=..., z=floor_z, reference_frame=handle.root)
on_floor.expression.limit(1000)
show_points(on_floor.tolist(backend=ProbabilisticBackend()))

The cloud collapsed onto the floor — but it still runs through cupboards and walls. Which floor points can a robot actually occupy? That is geometry again, so derive it from the world: a map of the navigable free space around the handle.

In [ ]:
free_space_map = navigation_map_at_target(handle.root, bloat_obstacles=0.5, search_range_x=5, search_range_y=5)
free_space_map.plot_and_show_free_space()

The straightforward way to use the map: sample as before, then keep only the points that fall into it. The keep-or-drop test is a **predicate** — a small class with two duties: *decide* (`__call__`) and *explain itself* (`_verbalization_fragment_`, the sentence used whenever a query containing it is verbalized).

✏️ **Your turn.** Fill in both. `free_space.node_of_point(point)` returns the region a point falls into, or `None`. For the sentence, aim for *"⟨free_space⟩ contains ⟨point⟩"*.

In [ ]:
@dataclass(eq=False)
class IsFree(Predicate):
    """
    Holds when ``point`` lies in the navigable free space ``free_space``.
    """

    free_space: GraphOfBoundingBoxes
    point: Point3

    def __call__(self) -> bool:
        ...

    @classmethod
    def _verbalization_fragment_(cls, fields: RenderedFields) -> VerbalizationFragment:
        return clause(...)

In [ ]:
@dataclass(eq=False)
class IsFree(Predicate):
    """
    Holds when ``point`` lies in the navigable free space ``free_space``.
    """

    free_space: GraphOfBoundingBoxes
    point: Point3

    def __call__(self) -> bool:
        return self.free_space.node_of_point(self.point) is not None

    @classmethod
    def _verbalization_fragment_(cls, fields: RenderedFields) -> VerbalizationFragment:
        return clause(Noun(fields["free_space"]), Verb("contain"), Noun(fields["point"]))

Now chain the two steps: generate points on the floor, then keep the free ones — and let the query explain the whole arrangement back to you, your own predicate included.

In [ ]:
on_floor = a(Point3)(x=..., y=..., z=floor_z, reference_frame=handle.root)
on_floor.expression.limit(1000)

standing_spot = a(Point3).from_(on_floor.evaluate(backend=ProbabilisticBackend()))
standing_spot.where(IsFree(free_space_map, standing_spot.variable))

spots = standing_spot.tolist()
show_points(spots)
print(verbalize_expression(standing_spot))
print(f"{len(spots)} of 1000 sampled points survived")

That worked, but look at the count: most of the 1000 samples were thrown away. This is **rejection sampling**, and we can do better — hand the map to the *sampler itself*, so it never proposes a blocked point in the first place. The free space is a union of boxes, i.e. plain ranges over `x` and `y`, which is exactly the kind of condition a probabilistic model can honor directly. `translate_free_space_to_where_condition` builds that condition. Start from a fresh generator:

In [ ]:
standing_spot = a(Point3)(x=..., y=..., z=floor_z, reference_frame=handle.root)

✏️ **Your turn.** The translation needs to know *which query variable* the condition should constrain. Which one?

In [ ]:
standing_spot = standing_spot.where(translate_free_space_to_where_condition(free_space_map.free_space_event, ...))

In [ ]:
standing_spot = standing_spot.where(translate_free_space_to_where_condition(free_space_map.free_space_event, standing_spot.variable))

In [ ]:
standing_spot.expression.limit(1000)

spots = standing_spot.tolist(backend=ProbabilisticBackend())
show_points(spots)
print(f"{len(spots)} of 1000 sampled points survived")

Every sample counts now. A query this size no longer reads well as one sentence, though — switch the verbalization to its hierarchical rendering, which indents the query like an outline, one branch per box of free space:

In [ ]:
VerbalizationPipeline.html(hierarchical=True).display(standing_spot)

The sampler has no taste: any free spot is as good as any other. Time to give the robot a preference of your own, as one more condition. Once more, start fresh:

In [ ]:
standing_spot = a(Point3)(x=..., y=..., z=floor_z, reference_frame=handle.root)

✏️ **Your turn — open ended.** State your preference as a condition on `standing_spot.variable` — for example `standing_spot.variable.x > 0`, which keeps the robot on the opening side of the door.

In [ ]:
preference = ...

In [ ]:
preference = standing_spot.variable.x > 0

Re-run the sampling with your preference in the `where` — and watch it appear in both RViz and the query's own account of itself.

In [ ]:
standing_spot.where(
    translate_free_space_to_where_condition(free_space_map.free_space_event, standing_spot.variable),
    preference,
)
standing_spot.expression.limit(1000)

spots = standing_spot.tolist(backend=ProbabilisticBackend())
show_points(spots)
VerbalizationPipeline.html(hierarchical=True).display(standing_spot)

## 6 · Go

Generated results come back **best first** — the samples are ordered by how likely the model considers them — so `spots[0]` is the place to stand. Time to act: spawn the robot, park its arms, and drive it there. Keep an eye on RViz.

In [ ]:
clear_points()
robot = spawn_pr2()

navigation_target = Pose(
    position=spots[0],
    orientation=Quaternion.from_rpy(roll=0.0, pitch=0.0, yaw=np.pi),
    reference_frame=handle.root,
)

plan = sequential(
    [ParkArmsAction(Arms.BOTH), NavigateAction(target_location=navigation_target)],
    context=Context.from_world(world),
).plan

with simulated_robot:
    plan.perform()

## 7 · Remember this morning

Robots that keep records of their experiences can learn from them later; in robotics such an episodic memory is called a **NEEM**. In EQL, memory is simply *another backend*: the same queries you have been writing against the live kitchen also run against a database of past experience. Create an empty long-term memory and store the morning's plan in it:

In [ ]:
long_term_memory = make_long_term_memory()

with long_term_memory.session_maker() as session:
    session.add(to_dao(plan))
    session.commit()

Now ask the memory what happened. Note what changed compared to every query so far: only the backend.

In [ ]:
plan_query = the(Plan)
VerbalizationPipeline.html(hierarchical=True).display(plan_query, backend=long_term_memory)

remembered = plan_query.tolist(backend=long_term_memory)
episode = remembered[0].root
print(f"Remembered: a plan that {episode.status.name}, "
      f"taking {(episode.end_time - episode.start_time).total_seconds():.1f} seconds")

✏️ **Your turn.** The plan did not go into memory alone — every body the robot experienced went in with it. Query the long-term memory for every `Body` it holds.

In [ ]:
body_query = ...

In [ ]:
body_query = a(Body)

In [ ]:
remembered_bodies = body_query.tolist(backend=long_term_memory)
print(f"{len(remembered_bodies)} bodies remembered")

A memory you can only look at is an archive. What makes this one *episodic* is that it comes back to life: `from_dao()` rebuilds the full `Plan` object, ready to be inspected, resumed, or learned from — memory becomes working memory again.

In [ ]:
reconstructed_plan: Plan = remembered[0].from_dao()
print(reconstructed_plan)

## Where you are now

In one robot morning you have used every core capability of EQL:

- **Finding** — querying the live world with `the` / `an`, variables, and `where` (part 2 and 3)
- **Extending** — your own `@symbolic_function`s and `Predicate`s, verbalization included (part 3 and 5)
- **Inferring** — rules that conclude new objects with `inference`, each answering for itself via `explain_inference` (part 4)
- **Generating** — underspecified `a(...)(x=..., ...)` statements answered by probabilistic models, and constraining them (part 5)
- **Remembering** — the identical queries against a NEEM-style long-term memory (part 7)

The kitchen, the model, and the memory are all still live below this cell — keep asking. A few ideas: ask the memory for the `NavigateAction` and read its target back, count the remembered bodies per type, or run any query above through `VerbalizationPipeline.plain()` and `.html(hierarchical=True)` to compare the renderings.

To go deeper, the EQL user guide lives in the [cognitive_robot_abstract_machine](https://github.com/AbdelrhmanBassiouny/cognitive_robot_abstract_machine) repository under `krrood/doc/eql`.

In [ ]:
# Your playground — the world, the sampler, and the memory are listening.